In [ ]:
import sys

sys.path.append("your_path")
import einops
import matplotlib.gridspec as gridspec
import numpy as np
import seaborn as sns
import torch
from matplotlib import pyplot as plt
from PIL import Image
from torch.nn import functional as F
from torchvision.transforms.v2 import (
    ColorJitter,
    Compose,
    RandomResizedCrop,
    RandomRotation,
    Resize,
    ToTensor,
)

from src.tasks.pretrain.utils import (
    butterworth_high_pass_filter,
    patchified_binary_mask,
    soft_edge_mask,
    thresholded_mask,
)

torch.manual_seed(123)
np.random.seed(123)

In [ ]:
# BEST_PARAMS = {
#     "cutoff": 62.03697363750256,
#     "order": 1,
#     "boundary_cut": 0.64534883655575285,
#     "th": 0.04797418066291606,
#     "sigma": 9.512161239925405,
# }

BEST_PARAMS = {
    "butterworth_cutoff": 14.0470391901718,
    "butterworth_order": 2,
    "soft_mask_th": 0.0868832016680601,
    "soft_mask_sigma": 10.133165328972893,
    "boundary_cutoff": 0.6701278573157876,
    "blur_sigma": 0.5185059445302515,
    "blur_radius": 1,
    "mask_ratio": 0.2506265193179896,
}

# Main Function

In [ ]:
def visualize_high_freq_filtering(
    img,
    fov_mask,
    cutoff=30,
    order=2,
    boundary_cut=0.5,
    blur_sigma=1.0,
    blur_radius=3,
    mask_ratio=0.15,
):
    g_channel = img[:, 1:2]
    g_high_pass, fshift_filtered, img_norm = butterworth_high_pass_filter(
        g_channel,
        fov_mask=fov_mask,
        cutoff=cutoff,
        order=order,
        boundary_cut=boundary_cut,
        blur_sigma=blur_sigma,
        blur_radius=blur_radius,
    )
    B, C, H, W = g_high_pass.shape

    g_high_pass_boundary_proc = g_high_pass
    # num_pixels = H * W
    # num_th = int(num_pixels * 0.01)
    # clip_val = torch.sort(g_high_pass_boundary_proc.view(B, -1), dim=1, descending=True)[0][
    #     :, num_th
    # ][:, None]
    # g_high_pass_boundary_proc = torch.where(
    #     g_high_pass_boundary_proc.view(B, -1) > clip_val,
    #     clip_val,
    #     g_high_pass_boundary_proc.view(B, -1),
    # ).reshape(B, C, H, W)

    g_high_pass_boundary_proc = (
        g_high_pass_boundary_proc - g_high_pass_boundary_proc.amin(dim=(-2, -1), keepdim=True)
    ) / (
        g_high_pass_boundary_proc.amax(dim=(-2, -1), keepdim=True)
        - g_high_pass_boundary_proc.amin(dim=(-2, -1), keepdim=True)
        + 1e-6
    )
    g_high_pass_boundary_proc = g_high_pass_boundary_proc.squeeze(1).cpu().numpy()
    g_high_pass_boundary_proc_colored = torch.stack(
        [
            torch.from_numpy(plt.cm.inferno(g_high_pass_boundary_proc[i])[:, :, :3])
            .permute(2, 0, 1)
            .float()
            for i in range(len(g_high_pass_boundary_proc))
        ],
        dim=0,
    )  # (B,3,H,W)

    fshift_filtered = fshift_filtered.abs()
    fshift_filtered = (fshift_filtered - fshift_filtered.amin(dim=(-2, -1), keepdim=True)) / (
        fshift_filtered.amax(dim=(-2, -1), keepdim=True)
        - fshift_filtered.amin(dim=(-2, -1), keepdim=True)
        + 1e-6
    )
    fshift_filtered_np = fshift_filtered.squeeze(1).cpu().numpy()
    fshift_filtered_colored = torch.stack(
        [
            torch.from_numpy(plt.cm.inferno(fshift_filtered_np[i])[:, :, :3])
            .permute(2, 0, 1)
            .float()
            for i in range(len(fshift_filtered_np))
        ],
        dim=0,
    )  # (B,3,H,W)

    g_high_mask_95_agv_pool_14_binary = patchified_binary_mask(
        g_high_pass, patch_size=14, mask_ratio=mask_ratio
    )
    g_high_mask_95_agv_pool_14_binary = F.interpolate(
        g_high_mask_95_agv_pool_14_binary.unsqueeze(1),
        size=(img.shape[-2], img.shape[-1]),
        mode="nearest",
    )

    # 이미지들을 하나로 합치기
    concat_img = torch.cat(
        [
            img,
            fshift_filtered_colored,
            g_high_pass_boundary_proc_colored,
            g_high_mask_95_agv_pool_14_binary.repeat(1, 3, 1, 1),
        ],
        dim=3,
    )
    # B,3,H,W -> 3,B*H,W 형태로 변환
    concat_img = einops.rearrange(concat_img, "b c h w -> c (b h) w")

    # 시각화
    plt.figure(figsize=(20, 20))
    plt.imshow(concat_img.permute(1, 2, 0).cpu())
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
img_path_list = [
    # "your_path/src/tasks/pretrain/visualization/imgs/E-ophtha_N.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/E-ophtha_N2.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_EDR.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_EDR2.png",
    "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_DAMD.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_DAMD2.png",
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_ADR.png",
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_ADR2.png",
    "your_path/DATASET/IMGNET/1K/val/ILSVRC2012_val_00000003.JPEG",
    "your_path/DATASET/IMGNET/1K/val/ILSVRC2012_val_00024989.JPEG",
    "your_path/DATASET/IMGNET/1K/val/ILSVRC2012_val_00049988.JPEG",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_WAMD.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_WAMD2.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_CRVO.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_CRVO2.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_BRVO.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/RFMiD_BRVO2.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/CHAKSU_GL.png",
    # "your_path/src/tasks/pretrain/visualization/imgs/drishtiGS_GL.png",
]
img = []
for img_path in img_path_list:
    im = ToTensor()(Image.open(img_path))
    if im.max() > 1:
        im = im / 255.0

    im = Resize((224, 224))(im)
    img.append(im)

img = torch.stack(img, dim=0)
fov_mask = soft_edge_mask(
    img, threshold_frac=BEST_PARAMS["soft_mask_th"], sigma=BEST_PARAMS["soft_mask_sigma"]
)
visualize_high_freq_filtering(
    img,
    fov_mask,
    cutoff=BEST_PARAMS["butterworth_cutoff"],
    order=BEST_PARAMS["butterworth_order"],
    boundary_cut=BEST_PARAMS["boundary_cutoff"],
    blur_sigma=BEST_PARAMS["blur_sigma"],
    blur_radius=BEST_PARAMS["blur_radius"],
    mask_ratio=BEST_PARAMS["mask_ratio"],
)

In [ ]:
img_path = (
    "your_path/src/tasks/pretrain/visualization/imgs/IDRiD_ADR2.png"
)
im = ToTensor()(Image.open(img_path))
if im.max() > 1:
    im = im / 255.0

im = Resize((224, 224))(im)
img = im.unsqueeze(0)
fov_mask = soft_edge_mask(
    img, threshold_frac=BEST_PARAMS["soft_mask_th"], sigma=BEST_PARAMS["soft_mask_sigma"]
)

g_ch = img[:, 1:2]
g_high_pass, fshift_filtered, img_norm = butterworth_high_pass_filter(
    g_ch,
    fov_mask=fov_mask,
    cutoff=BEST_PARAMS["butterworth_cutoff"],
    order=BEST_PARAMS["butterworth_order"],
    boundary_cut=BEST_PARAMS["boundary_cutoff"],
    blur_sigma=BEST_PARAMS["blur_sigma"],
    blur_radius=BEST_PARAMS["blur_radius"],
)

B, C, H, W = g_high_pass.shape

g_high_pass_boundary_proc = g_high_pass
# num_pixels = H * W
# num_th = int(num_pixels * 0.01)
# clip_val = torch.sort(g_high_pass_boundary_proc.view(B, -1), dim=1, descending=True)[0][:, num_th][
#     :, None
# ]
# g_high_pass_boundary_proc = torch.where(
#     g_high_pass_boundary_proc.view(B, -1) > clip_val,
#     clip_val,
#     g_high_pass_boundary_proc.view(B, -1),
# ).reshape(B, C, H, W)

g_high_pass_boundary_proc = (
    g_high_pass_boundary_proc - g_high_pass_boundary_proc.amin(dim=(-2, -1), keepdim=True)
) / (
    g_high_pass_boundary_proc.amax(dim=(-2, -1), keepdim=True)
    - g_high_pass_boundary_proc.amin(dim=(-2, -1), keepdim=True)
    + 1e-6
)
g_high_pass_boundary_proc = g_high_pass_boundary_proc.squeeze(1).cpu().numpy()
IFFT = torch.stack(
    [
        torch.from_numpy(plt.cm.inferno(g_high_pass_boundary_proc[i])[:, :, :3])
        .permute(2, 0, 1)
        .float()
        for i in range(len(g_high_pass_boundary_proc))
    ],
    dim=0,
)  # (B,3,H,W)

fshift_filtered = fshift_filtered.abs()
fshift_filtered = (fshift_filtered - fshift_filtered.amin(dim=(-2, -1), keepdim=True)) / (
    fshift_filtered.amax(dim=(-2, -1), keepdim=True)
    - fshift_filtered.amin(dim=(-2, -1), keepdim=True)
    + 1e-6
)
fshift_filtered_np = fshift_filtered.squeeze(1).cpu().numpy()
F_hp = torch.stack(
    [
        torch.from_numpy(plt.cm.inferno(fshift_filtered_np[i])[:, :, :3]).permute(2, 0, 1).float()
        for i in range(len(fshift_filtered_np))
    ],
    dim=0,
)  # (B,3,H,W)

M_H = patchified_binary_mask(g_high_pass, patch_size=14, mask_ratio=BEST_PARAMS["mask_ratio"])
M_H_intp = F.interpolate(
    M_H.unsqueeze(1),
    size=(img.shape[-2], img.shape[-1]),
    mode="nearest",
)

In [ ]:
# 이미지들을 하나로 합치기
concat_img = torch.cat(
    [
        # img_masked,
        img,
        F_hp,
        IFFT,
        M_H_intp.repeat(1, 3, 1, 1),
    ],
    dim=3,
)
# B,3,H,W -> 3,B*H,W 형태로 변환
concat_img = einops.rearrange(concat_img, "b c h w -> c (b h) w")

# 시각화
plt.figure(figsize=(20, 20))
plt.imshow(concat_img.permute(1, 2, 0).cpu())
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 마스크에서 1인 부분을 찾아서 절반을 0.5로 변경
mask_ones = M_H == 1
num_ones = mask_ones.sum()
num_to_change = num_ones // 2

# 1인 위치들을 랜덤하게 선택하여 절반을 0.5로 변경
if num_to_change > 0:
    ones_indices = torch.where(mask_ones)
    random_indices = torch.randperm(len(ones_indices[0]))[:num_to_change]

    # 각 차원의 인덱스를 개별적으로 선택하여 인덱싱
    selected_indices = tuple(ones_indices[i][random_indices] for i in range(len(ones_indices)))
    M_H[selected_indices] = 0.5

M_H.shape

In [ ]:
M_H_intp2 = F.interpolate(
    M_H.unsqueeze(1),
    size=(img.shape[-2], img.shape[-1]),
    mode="nearest",
)
img_masked = img.clone()

# 마스크를 3채널로 확장
mask_3ch = M_H_intp2.repeat(1, 3, 1, 1)

# 0.5인 부분을 노란색으로 변경
yellow_mask = mask_3ch == 0.5
# 마스크를 각 채널별로 적용
img_masked[0, 0, yellow_mask[0, 0]] = 230 / 255  # R 채널
img_masked[0, 1, yellow_mask[0, 1]] = 230 / 255  # G 채널
img_masked[0, 2, yellow_mask[0, 2]] = 230 / 255  # B 채널

# 0인 부분을 초록색으로 변경
zero_mask = mask_3ch == 0
# 마스크를 각 채널별로 적용
img_masked[0, 0, zero_mask[0, 0]] = 0  # R 채널
img_masked[0, 1, zero_mask[0, 1]] = 0  # G 채널
img_masked[0, 2, zero_mask[0, 2]] = 0  # B 채널

In [ ]:
# 이미지들을 PNG 파일로 저장
import os

import torchvision.transforms as transforms
from PIL import Image

# 저장할 디렉토리 생성
save_dir = "your_path/src/tasks/pretrain/visualization/feats/save"
os.makedirs(save_dir, exist_ok=True)

# 텐서를 PIL 이미지로 변환하는 함수


def tensor_to_pil(tensor):
    """텐서를 PIL 이미지로 변환"""
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)  # B 차원 제거

    # 텐서를 0-1 범위로 정규화
    if tensor.min() < 0 or tensor.max() > 1:
        tensor = (tensor - tensor.min()) / (tensor.max() - tensor.min())

    # 텐서를 numpy로 변환하고 차원 순서 조정
    img_np = tensor.permute(1, 2, 0).cpu().numpy()

    # 0-1 범위를 0-255로 변환
    img_np = (img_np * 255).astype("uint8")

    return Image.fromarray(img_np)


# 단일 채널 텐서를 PIL 이미지로 변환하는 함수


def single_channel_tensor_to_pil(tensor):
    """단일 채널 텐서를 PIL 이미지로 변환"""
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)  # B 차원 제거
    if tensor.dim() == 3:
        tensor = tensor.squeeze(0)  # C 차원 제거

    # 텐서를 0-1 범위로 정규화
    if tensor.min() < 0 or tensor.max() > 1:
        tensor = (tensor - tensor.min()) / (tensor.max() - tensor.min())

    # 텐서를 numpy로 변환
    img_np = tensor.cpu().numpy()

    # 0-1 범위를 0-255로 변환
    img_np = (img_np * 255).astype("uint8")

    return Image.fromarray(img_np, mode="L")


# 각 이미지 저장
tensor_to_pil(img).save(os.path.join(save_dir, "original_image.png"))
tensor_to_pil(F_hp).save(os.path.join(save_dir, "high_frequency.png"))
tensor_to_pil(IFFT).save(os.path.join(save_dir, "inverse_fft.png"))
single_channel_tensor_to_pil(M_H_intp).save(os.path.join(save_dir, "mask_interpolated.png"))
tensor_to_pil(img_masked).save(os.path.join(save_dir, "masked_image.png"))

# g_ch는 단일 채널이므로 별도 처리
single_channel_tensor_to_pil(g_ch.squeeze(1)).save(os.path.join(save_dir, "g_ch.png"))

print(f"모든 이미지가 '{save_dir}' 디렉토리에 저장되었습니다.")

In [ ]:
g_high_pass, fshift_filtered, img_norm = butterworth_high_pass_filter(
    g_ch,
    fov_mask=fov_mask,
    cutoff=BEST_PARAMS["butterworth_cutoff"],
    order=BEST_PARAMS["butterworth_order"],
    boundary_cut=BEST_PARAMS["boundary_cutoff"],
    blur_sigma=BEST_PARAMS["blur_sigma"],
    blur_radius=BEST_PARAMS["blur_radius"],
    normalize=False,
)
print(g_high_pass.min(), g_high_pass.max())